In [ ]:
import sys
import logging
from pprint import pprint

from tqdm import tqdm
import pyarrow as pa

In [ ]:
sys.path.append("../")

from src.data import BIERDataset
from src.chunking import TokenChunker
from src.embeddings import Embedder
from src.index import Indexer
from src.retrieval import Retriever
from src.evaluation import Evaluator

logger = logging.getLogger(__name__)
logging.basicConfig(level=logging.INFO)

In [ ]:
dataset_name = 'scifact'
vector_db_uri = 'lancedb_index'
path_qrel = f"../datasets/beir/datasets/{dataset_name}/qrels/train.tsv"
path_corpus = f"../datasets/beir/datasets/{dataset_name}/corpus.jsonl"
path_queries = f"../datasets/beir/datasets/{dataset_name}/queries.jsonl"

dataset = BIERDataset()
chunker = TokenChunker(
    chunk_size=512,
    chunk_overlap=50,
    encoding='cl100k_base'
)
embedder = Embedder()
indexer = Indexer(uri=vector_db_uri)
evaluator = Evaluator()


In [ ]:
# Process documents
documents_loader = dataset.document_loader(file_path=path_corpus, batch_size=512)

# Stream documents in batches
db_mode = "overwrite"
for idx, batch in tqdm(enumerate(documents_loader)):

    # Chunk
    chunks = chunker.chunk(documents=batch)
    logger.debug(f"chunking docs with ID: {[c.doc_id for c in chunks]}")


    # Embedding
    embeddings = embedder.embed_chunks(chunks)
    logger.debug(f"embedding chunks with doc_id: {[e.chunk.doc_id for e in embeddings]}")


    # Add to index
    indexer.add_to_index(embeddings, dataset_name=dataset_name, mode=db_mode)
    db_mode = "append"
    logger.debug(f'added batch {idx} to index')

In [ ]:
indexer.get_index().open_table(dataset_name).to_pandas().head()

In [ ]:
# Retrieval
retrievals = []
query_loader = dataset.query_loader(path_queries, batch_size=128)
retriever = Retriever(vector_db_uri)

# Stream queries in batches.
for _, batch in tqdm(enumerate(query_loader)):
    embeddings = embedder.embed_queries(batch)
    batch_results = retriever.search(
        dataset_name=dataset_name, q_embeddings=embeddings, dist_metric="cosine", top_k=10
    )
    retrievals.append(batch_results)

retrievals = pa.concat_tables(retrievals)
retrievals_df = retrievals.to_pandas()

In [ ]:
# Evaluation
# Convert to pytrec_eval format
run = evaluator.df_to_pytrec(retrievals_df)

with open(path_qrel, "r") as f_qrel:
    qrel = evaluator.parse_qrel(f_qrel, ignore_header=True)

# evaluate
measures = [
    "map_cut.1",
    "ndcg_cut.1",
    "recall.1",
    "P.1",
    "map_cut.5",
    "ndcg_cut.5",
    "recall.5",
    "P.5",
    "map_cut.10",
    "ndcg_cut.10",
    "recall.10",
    "P.10",
]

results = evaluator.evaluate(qrel, run, measures)
logger.info("Evalutation completed!")